In [2]:
import urllib.request
url = (
    "https://raw.githubusercontent.com/rasbt/"
    "LLMs-from-scratch/refs/heads/main/ch07/"
    "01_main-chapter-code/gpt_download.py"
)
filename = url.split('/')[-1]
urllib.request.urlretrieve(url, filename)

url = (
    "https://raw.githubusercontent.com/rasbt/"
    "LLMs-from-scratch/refs/heads/main/ch07/"
    "01_main-chapter-code/previous_chapters.py"
)
filename = url.split('/')[-1]
urllib.request.urlretrieve(url, filename)

url = (
    "https://raw.githubusercontent.com/rasbt"
    "/LLMs-from-scratch/refs/heads/main/ch07/"
    "01_main-chapter-code/instruction-data-with-response.json"
)
filename = url.split('/')[-1]
urllib.request.urlretrieve(url, filename)

('instruction-data-with-response.json',
 <http.client.HTTPMessage at 0x7da1d31253d0>)

In [3]:
# This cell is optional; it allows you to restart the notebook
# and only run section 7.7 without rerunning any of the previous code
import json
from tqdm import tqdm

file_path = "instruction-data-with-response.json"

with open(file_path, "r") as file:
    test_data = json.load(file)

def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""

    return instruction_text + input_text

In [4]:
from google.colab import userdata
API_KEY = userdata.get('si_API_KEY')

In [5]:
from openai import OpenAI

def query_model(
    prompt,
    model="deepseek-ai/deepseek-v4-pro",
    url="https://api.siliconflow.cn/v1",
    max_retries=5,
    timeout=20  # 超时秒数
):
    client = OpenAI(
    api_key=API_KEY,
    base_url=url,
    timeout=timeout,
    )
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
                max_tokens=2048,  # 只返回分数，不需要2048
                seed=123
            )
            return response.choices[0].message.content

        except Exception as e:
            wait = 2 ** attempt  # 指数退避：1s, 2s, 4s
            print(f"\nAttempt {attempt+1} failed: {e}. Retrying in {wait}s...")
            time.sleep(wait)

    print(f"\nAll {max_retries} attempts failed, skipping.")
    return None

In [6]:
print(query_model("你是谁"))

你好！我是DeepSeek，由深度求索公司创造的AI助手。😊

我是一个纯文本模型，可以帮你解答问题、处理信息、进行对话交流。虽然我不支持多模态识别（比如直接识别图片内容），但我可以：

- 📚 阅读你上传的文件（图片、PDF、Word、Excel、PPT等），从中提取文字信息
- 🔗 阅读链接内容
- 🌐 联网搜索（需要你在Web/App端手动开启）
- 💬 处理超长上下文（1M tokens，相当于能一次性处理《三体》三部曲那么多内容！）

我的知识截止到2025年5月，完全免费使用，App端还支持语音输入。你可以通过官方应用商店下载我的App。

有什么我可以帮你的吗？尽管问我！✨


In [7]:
for entry in test_data[:3]:
    prompt = (
        f"Given the input `{format_input(entry)}` "
        f"and correct output `{entry['output']}`, "
        f"score the model response `{entry['model_response']}`"
        f" on a scale from 0 to 100, where 100 is the best score. "
    )
    print("\nDataset response:")
    print(">>", entry['output'])
    print("\nModel response:")
    print(">>", entry["model_response"])
    print("\nScore:")
    print(">>", query_model(prompt))
    print("\n-------------------------")


Dataset response:
>> The car is as fast as lightning.

Model response:
>> The car is as fast as a bullet.

Score:
>> The model response correctly rewrites the sentence using a simile, replacing "very fast" with "as fast as a bullet," which is a common and appropriate simile for speed. It fully satisfies the instruction, so it merits a perfect score.

**Score: 100**

-------------------------

Dataset response:
>> The type of cloud typically associated with thunderstorms is cumulonimbus.

Model response:
>> The type of cloud associated with thunderstorms is a cumulus cloud.

Score:
>> 0

-------------------------

Dataset response:
>> Jane Austen.

Model response:
>> The author of 'Pride and Prejudice' is Jane Austen.

Score:
>> 100

-------------------------


In [8]:
def generate_model_scores(json_data, json_key, model="deepseek-ai/deepseek-v4-pro"):
    scores = []
    for entry in tqdm(json_data, desc="Scoring entries"):
        prompt = (
            f"Given the input `{format_input(entry)}` "
            f"and correct output `{entry['output']}`, "
            f"score the model response `{entry[json_key]}`"
            f" on a scale from 0 to 100, where 100 is the best score. "
            f"Respond with the integer number only."
        )
        score = query_model(prompt, model)
        try:
            scores.append(int(score))
        except ValueError:
            print(f"Could not convert score: {score}")
            continue

    return scores

scores = generate_model_scores(test_data, "model_response")
print(f"Number of scores: {len(scores)} of {len(test_data)}")
print(f"Average score: {sum(scores)/len(scores):.2f}\n")

Scoring entries: 100%|██████████| 110/110 [16:07<00:00,  8.79s/it]

Number of scores: 110 of 110
Average score: 30.52

